# Benchmark — Google Colab

**Upload lên Drive một lần duy nhất:**

```
MyDrive/BKAI/
├── embeddings/
│   └── bge-large-v1.5/        ← upload folder này (~280 MB)
│       ├── combined_features.npy
│       ├── profile_embeddings.npy
│       ├── mood_vectors.npy
│       ├── movie_id_index.json
│       └── ...
└── data/
    └── processed/             ← upload folder này (~238 MB)
        ├── train.csv
        ├── val.csv
        ├── test.csv
        ├── item_map.json
        ├── user_map.json
        └── stats.json
```

**Mỗi lần chạy:** Colab tự clone code từ GitHub — chỉ cần chạy notebook từ trên xuống.

## 0. Cấu hình

In [ ]:
# ── GitHub ───────────────────────────────────────────────────────────────────
GITHUB_REPO    = 'https://github.com/H1epVu/benchmark_20M.git'
COLAB_CODE_DIR = '/content/benchmark'

# ── Drive — data (upload 1 lần) ───────────────────────────────────────────────
DRIVE_EMBEDDING_DIR = '/content/drive/MyDrive/BKAI/embeddings/bge-large-v1.5'
DRIVE_DATA_DIR      = '/content/drive/MyDrive/BKAI/embeddings/data/processed'

# ── Drive — output (checkpoints & results tự động lưu lên đây) ───────────────
DRIVE_OUTPUT_DIR    = '/content/drive/MyDrive/BKAI/benchmark_output'

BENCHMARK_DIR = COLAB_CODE_DIR

print('Config OK')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone code từ GitHub

Nếu đã clone trong session này thì `git pull` để lấy code mới nhất.

In [ ]:
import os

if os.path.exists(COLAB_CODE_DIR):
    print('Repo đã tồn tại — pulling latest code...')
    !git -C {COLAB_CODE_DIR} pull
else:
    print('Cloning repo...')
    !git clone {GITHUB_REPO} {COLAB_CODE_DIR}

print(f'[OK] Code tại {COLAB_CODE_DIR}')

## 3. Link checkpoint & results lên Drive

Checkpoints và results được ghi trực tiếp lên Drive — không mất khi Colab hết session.

In [ ]:
import os, sys

# Set EMBEDDING_DIR env var → config.py sẽ đọc từ đây thay vì đường dẫn mặc định
os.environ['EMBEDDING_DIR'] = DRIVE_EMBEDDING_DIR

# Trỏ DATA_DIR về Drive qua symlink
data_link = os.path.join(COLAB_CODE_DIR, 'data', 'processed')
os.makedirs(os.path.dirname(data_link), exist_ok=True)

# Xóa symlink cũ nếu tồn tại (kể cả broken symlink mà os.path.exists() bỏ sót)
if os.path.islink(data_link):
    os.unlink(data_link)

if not os.path.exists(data_link):
    os.symlink(DRIVE_DATA_DIR, data_link)
    print(f'[LINK] data/processed/ → {DRIVE_DATA_DIR}')
else:
    print(f'[OK]   data/processed/ đã được link')

# Kiểm tra
print(f'\n=== Embeddings ({DRIVE_EMBEDDING_DIR}) ===')
for f in ['combined_features.npy', 'profile_embeddings.npy', 'mood_vectors.npy', 'movie_id_index.json']:
    status = 'OK' if os.path.exists(f'{DRIVE_EMBEDDING_DIR}/{f}') else 'THIẾU'
    print(f'  [{status}] {f}')

print(f'\n=== Data processed ({DRIVE_DATA_DIR}) ===')
for f in ['train.csv', 'val.csv', 'test.csv', 'item_map.json', 'stats.json']:
    status = 'OK' if os.path.exists(f'{DRIVE_DATA_DIR}/{f}') else 'THIẾU'
    print(f'  [{status}] {f}')

## 4. Link checkpoint & results thẳng lên Drive

Mọi file `.pt` được ghi trực tiếp lên Drive trong lúc training.

In [ ]:
import os, shutil

# Checkpoints và results ghi thẳng lên Drive qua symlink
OUTPUT_LINKS = [
    (f'{BENCHMARK_DIR}/checkpoints', f'{DRIVE_OUTPUT_DIR}/checkpoints'),
    (f'{BENCHMARK_DIR}/results',     f'{DRIVE_OUTPUT_DIR}/results'),
]

for link_path, target_path in OUTPUT_LINKS:
    os.makedirs(target_path, exist_ok=True)
    if os.path.exists(link_path) and not os.path.islink(link_path):
        shutil.rmtree(link_path)
    if not os.path.islink(link_path):
        os.symlink(target_path, link_path)
        print(f'[LINK] {link_path}\n       → {target_path}')
    else:
        print(f'[OK]   {link_path}')

## 5. Cài đặt dependencies

In [ ]:
%pip install -q torch numpy pandas scipy scikit-learn sentence-transformers tqdm pyyaml

## 6. Kiểm tra GPU & môi trường

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Không có GPU — vào Runtime > Change runtime type > GPU để bật')

## 7. Kiểm tra dữ liệu & embeddings

In [ ]:
from pathlib import Path

print('=== Drive symlinks ===')
for name, path in [('checkpoints', Path(BENCHMARK_DIR) / 'checkpoints'),
                   ('results',     Path(BENCHMARK_DIR) / 'results'),
                   ('data/processed', Path(BENCHMARK_DIR) / 'data' / 'processed')]:
    if path.is_symlink():
        print(f'  [LINK] {name}/ → {path.resolve()}')
    else:
        print(f'  [WARN] {name}/ chưa được link — chạy lại cell 3 & 4')

## 8. Setup Python path

In [ ]:
import sys
sys.path.insert(0, BENCHMARK_DIR)

# EMBEDDING_DIR đã được set ở cell 3 — import config sau khi set env var
from config import EMBED_DIM, NUM_EPOCHS, PATIENCE, FEATURE_CONFIGS, MAX_SEQ_LEN, SEQ_BATCH_SIZE
print(f'EMBED_DIM={EMBED_DIM}, MAX_SEQ_LEN={MAX_SEQ_LEN}, SEQ_BATCH_SIZE={SEQ_BATCH_SIZE}')
print(f'Feature configs: {list(FEATURE_CONFIGS.keys())}')

## 9. Chạy một thí nghiệm đơn

**CF models (LightGCN, BPR-MF, ...):**

| Tham số | Các lựa chọn |
|---|---|
| `MODEL` | `bpr_mf`, `lightgcn`, `lightgcn_sf`, `xsimgcl`, `simgcl`, `lightgcl`, `kar` |
| `FEATURES` | `none`, `genome`, `bert_title`, `llm_profile`, `llm_mood`, `llm_themes`, `llm_prof_mood`, `llm_all`, `genome_llm` |

**Sequential models — Direction 4 (SASRec):**

| Tham số | Các lựa chọn |
|---|---|
| `MODEL` | `sasrec` |
| `FEATURES` | `none` (baseline) hoặc `llm_prof_mood` (M7) |
| `INJECTION` | `none`, `input`, `ffn`, `output`, `input+ffn`, `input+output`, `ffn+output`, `all` |
| `SEQ_BATCH_SIZE` | T4 → `1024`, A100 → `2048` |

In [ ]:
import os
os.chdir(str(BENCHMARK_DIR))

# ── Tham số ──────────────────────────────────────────────────────────────
MODEL          = 'sasrec'         # đổi model ở đây
DATASET        = 'ml20m'          # ml20m | sub_ml20m | ml1m | amazon
FEATURES       = 'llm_prof_mood'  # 'none' = baseline, 'llm_prof_mood' = M7
INJECTION      = 'input'          # chỉ dùng cho sasrec; bỏ qua với CF models
SEED           = 42
EPOCHS         = 100
SEQ_BATCH_SIZE = 4096             # L4=4096, T4=1024, A100=8192

# ── Chạy ─────────────────────────────────────────────────────────────────
if MODEL == 'sasrec':
    !python run_experiment.py \
        --model {MODEL} \
        --dataset {DATASET} \
        --features {FEATURES} \
        --injection {INJECTION} \
        --seed {SEED} \
        --epochs {EPOCHS} \
        --seq-batch-size {SEQ_BATCH_SIZE} \
        --device auto
else:
    !python run_experiment.py \
        --model {MODEL} \
        --dataset {DATASET} \
        --features {FEATURES} \
        --seed {SEED} \
        --epochs {EPOCHS} \
        --device auto

## 10. Chạy Quick Test (tất cả model, 1 seed, 20 epochs)

In [ ]:
import os
os.chdir(str(BENCHMARK_DIR))
!python run_ablation.py --quick

## 11. Chạy R1 — RLMRec-plus

In [ ]:
import os
os.chdir(f'{COLAB_CODE_DIR}/benchmark/external/RLMRec')

SEED   = 42
EPOCHS = 100

!python encoder/train_encoder.py \
    --model lightgcn_plus \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 12. Chạy R2 — RLMRec-gene

In [ ]:
import os
os.chdir(f'{COLAB_CODE_DIR}/benchmark/external/RLMRec')

SEED   = 42
EPOCHS = 100

!python encoder/train_encoder.py \
    --model lightgcn_gene \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 13. Kiểm tra checkpoint đã lưu trên Drive

In [ ]:
import os

ckpt_dir = f'{DRIVE_OUTPUT_DIR}/checkpoints'
if os.path.exists(ckpt_dir):
    experiments = sorted(os.listdir(ckpt_dir))
    print(f'Checkpoint trên Drive ({len(experiments)} experiments):')
    for exp in experiments:
        files = os.listdir(f'{ckpt_dir}/{exp}')
        print(f'  {exp}/  {files}')
else:
    print('Chưa có checkpoint nào.')